# 05: Unified Comparison - ANNS Methods
## Thesis: Approximate Nearest Neighbor Search with Adaptive Search Strategies

**Objective**: Compare four HNSW-based methods:
1. **Plain HNSW** - Standard hierarchical navigable small world
2. **DARTH** - Dynamic Adaptive Re-termination using history features
3. **PiP** - Probabilistic pruning with saturation-based early termination
4. **Ada-ef** - Adaptive exploration factor based on query difficulty estimation

**Key Questions**:
- Which method gives the best recall-speed tradeoff?
- Which method reduces unnecessary search effort?
- Which method is most stable across the workload?

---

## 1. Imports and Path Setup

In [16]:
import sys
import os
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

PROJECT_ROOT = Path('/Users/Damian/approximate-nearest-neighbor-graphs')
sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_CSV = PROJECT_ROOT / 'results_csv'
PLOT_RESULTS = PROJECT_ROOT / 'plot_results'
DATASETS_DIR = PROJECT_ROOT / 'Datasets'

RESULTS_CSV.mkdir(exist_ok=True)
PLOT_RESULTS.mkdir(exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Results CSV dir: {RESULTS_CSV}")
print(f"Plot results dir: {PLOT_RESULTS}")

Project root: /Users/Damian/approximate-nearest-neighbor-graphs
Results CSV dir: /Users/Damian/approximate-nearest-neighbor-graphs/results_csv
Plot results dir: /Users/Damian/approximate-nearest-neighbor-graphs/plot_results


---

## 2. Load Dataset and Ground Truth

In [17]:
from utils.read_files import read_fvecs, read_ivecs

DATASET_NAME = 'siftsmall'

BASE_FILE = DATASETS_DIR / DATASET_NAME / f'{DATASET_NAME}_base.fvecs'
QUERY_FILE = DATASETS_DIR / DATASET_NAME / f'{DATASET_NAME}_query.fvecs'
GT_FILE = DATASETS_DIR / DATASET_NAME / f'{DATASET_NAME}_groundtruth.ivecs'

print(f"Loading dataset: {DATASET_NAME}")
print(f"Base file: {BASE_FILE}")
print(f"Query file: {QUERY_FILE}")
print(f"Ground truth file: {GT_FILE}")

xb = read_fvecs(str(BASE_FILE))
xq = read_fvecs(str(QUERY_FILE))
I_gt = read_ivecs(str(GT_FILE))

print(f"\nDataset shapes:")
print(f"  Base (xb): {xb.shape}")
print(f"  Query (xq): {xq.shape}")
print(f"  Ground truth (I_gt): {I_gt.shape}")
print(f"  Dimension: {xb.shape[1]}")

Loading dataset: siftsmall
Base file: /Users/Damian/approximate-nearest-neighbor-graphs/Datasets/siftsmall/siftsmall_base.fvecs
Query file: /Users/Damian/approximate-nearest-neighbor-graphs/Datasets/siftsmall/siftsmall_query.fvecs
Ground truth file: /Users/Damian/approximate-nearest-neighbor-graphs/Datasets/siftsmall/siftsmall_groundtruth.ivecs

Dataset shapes:
  Base (xb): (10000, 128)
  Query (xq): (100, 128)
  Ground truth (I_gt): (100, 100)
  Dimension: 128


---

## 3. Common Evaluation Utilities

In [13]:
from metrics.benchMark import recall_at_k

K = 10
WARMUP_RUNS = 2


def measure_search(search_fn, Xq, k, I_true=None, warmup=WARMUP_RUNS):
    """
    Standardized benchmark function for all methods.
    Ensures identical measurement protocol across methods.
    
    Args:
        search_fn: Callable that takes (Xq, k) and returns (D, I)
        Xq: Query vectors
        k: Number of nearest neighbors
        I_true: Ground truth indices (optional)
        warmup: Number of warmup runs
    
    Returns:
        dict with QPS, latency, recall metrics
    """
    if warmup > 0:
        for _ in range(warmup):
            _ = search_fn(Xq[:min(len(Xq), 64)], k)

    t0 = time.perf_counter()
    D, I = search_fn(Xq, k)
    t1 = time.perf_counter()

    total_s = t1 - t0
    qps = len(Xq) / total_s if total_s > 0 else float('inf')
    avg_latency_ms = (total_s / len(Xq)) * 1000
    recall = recall_at_k(I_true, I, k) if I_true is not None else np.nan

    return {
        'QPS': qps,
        'Avg Latency (ms)': avg_latency_ms,
        'Total Time (s)': total_s,
        f'Recall@{k}': recall
    }


def run_single_experiment(name, build_fn, search_fn, Xb, Xq, I_gt, k, **build_params):
    """
    Build index and run benchmark.
    
    Args:
        name: Method name for results
        build_fn: Index construction function
        search_fn: Search function factory
        Xb, Xq: Base and query vectors
        I_gt: Ground truth
        k: k for recall
        **build_params: Parameters for build_fn
    
    Returns:
        dict with all results including build time
    """
    t_build_start = time.perf_counter()
    index = build_fn(Xb, **build_params)
    t_build_end = time.perf_counter()
    build_time = t_build_end - t_build_start

    results = measure_search(search_fn, Xq, k, I_true=I_gt)
    results['Method'] = name
    results['Build Time (s)'] = build_time
    results['k'] = k

    for param_name, param_value in build_params.items():
        results[param_name] = param_value

    return results


def create_results_df(rows):
    """Create and format results DataFrame."""
    df = pd.DataFrame(rows)
    
    cols_order = ['Method', 'k']
    param_cols = [c for c in df.columns if c not in 
                  ['Method', 'k', 'QPS', 'Avg Latency (ms)', 'Total Time (s)', 
                   f'Recall@{K}', 'Build Time (s)']]
    metric_cols = ['QPS', 'Avg Latency (ms)', f'Recall@{K}', 'Total Time (s)', 'Build Time (s)']
    
    df = df[cols_order + param_cols + metric_cols]
    return df.sort_values(['Method', 'k'] + param_cols).reset_index(drop=True)

---

## 4. Import Algorithm Wrappers

In [14]:
from testing.comparing_algorithm import (
    build_hnsw_New, hnsw_New_search_fn,
    build_hnsw_darth, hnsw_darth_search_fn,
    build_hnsw_pip, hnsw_pip_search_fn,
    build_hnsw_adaef, hnsw_adaef_search_fn,
    DummyPredictor
)

print("Algorithm wrappers imported successfully")

Algorithm wrappers imported successfully


---

## 5. HNSW Baseline

In [15]:
print("Running HNSW Baseline experiments...")

hnsw_results = []

M_values = [16]
efC_values = [100, 200]
efS_values = [50, 100, 200]

for M in M_values:
    for efC in efC_values:
        for efS in efS_values:
            print(f"  HNSW: M={M}, efC={efC}, efS={efS}", end=' ')
            
            result = run_single_experiment(
                name='HNSW',
                build_fn=build_hnsw_New,
                search_fn=hnsw_New_search_fn,
                Xb=xb, Xq=xq, I_gt=I_gt,
                k=K,
                M=M, efC=efC
            )
            
            result['efSearch'] = efS
            result['efConstruction'] = efC
            result['M'] = M
            
            search_fn = hnsw_New_search_fn(result.get('index', None) or 
                                           build_hnsw_New(xb, M=M, efC=efC), efS=efS)
            search_fn = lambda Xq, k, idx_m=M, efC_m=efC, efS_m=efS: hnsw_New_search_fn(
                build_hnsw_New(xb, M=idx_m, efC=efC_m), efS_m)(Xq, k)
            
            final_result = {
                'Method': 'HNSW',
                'M': M,
                'efConstruction': efC,
                'efSearch': efS,
                'k': K
            }
            
            index = build_hnsw_New(xb, M=M, efC=efC)
            sf = hnsw_New_search_fn(index, efS=efS)
            metrics = measure_search(sf, xq, K, I_gt)
            
            final_result.update(metrics)
            hnsw_results.append(final_result)
            
            print(f"-> Recall@{K}={metrics[f'Recall@{K}']:.4f}, QPS={metrics['QPS']:.2f}")

hnsw_df = pd.DataFrame(hnsw_results)
print(f"\nHNSW experiments completed: {len(hnsw_df)} configurations")

Running HNSW Baseline experiments...
  HNSW: M=16, efC=100, efS=50 

AttributeError: module 'hnsw_cpp' has no attribute 'HNSWIndex'

---

## 6. DARTH

In [ ]:
print("Running DARTH experiments...")

darth_results = []

M_values = [16]
efC_values = [100, 200]
efS_values = [50, 100]
Rt_values = [0.90, 0.95]

for M in M_values:
    for efC in efC_values:
        for efS in efS_values:
            for Rt in Rt_values:
                print(f"  DARTH: M={M}, efC={efC}, efS={efS}, Rt={Rt}", end=' ')
                
                predictor = DummyPredictor()
                index = build_hnsw_darth(xb, M=M, efC=efC)
                sf = hnsw_darth_search_fn(index, efS=efS, Rt=Rt, predictor=predictor)
                metrics = measure_search(sf, xq, K, I_gt)
                
                result = {
                    'Method': 'DARTH',
                    'M': M,
                    'efConstruction': efC,
                    'efSearch': efS,
                    'Rt': Rt,
                    'k': K,
                    'ipi': 200,
                    'mpi': 20
                }
                result.update(metrics)
                darth_results.append(result)
                
                print(f"-> Recall@{K}={metrics[f'Recall@{K}']:.4f}, QPS={metrics['QPS']:.2f}")

darth_df = pd.DataFrame(darth_results)
print(f"\nDARTH experiments completed: {len(darth_df)} configurations")

---

## 7. PiP

In [ ]:
print("Running PiP experiments...")

pip_results = []

M_values = [16]
efC_values = [100, 200]
efS_values = [50, 100, 200]
pip_gamma_values = [95.0]
pip_delta_values = [20]

for M in M_values:
    for efC in efC_values:
        for efS in efS_values:
            for pip_gamma in pip_gamma_values:
                for pip_delta in pip_delta_values:
                    print(f"  PiP: M={M}, efC={efC}, efS={efS}, gamma={pip_gamma}, delta={pip_delta}", end=' ')
                    
                    index = build_hnsw_pip(xb, M=M, efC=efC, 
                                          pip_gamma=pip_gamma, pip_delta=pip_delta)
                    sf = hnsw_pip_search_fn(index, efS=efS)
                    metrics = measure_search(sf, xq, K, I_gt)
                    
                    result = {
                        'Method': 'PiP',
                        'M': M,
                        'efConstruction': efC,
                        'efSearch': efS,
                        'pip_gamma': pip_gamma,
                        'pip_delta': pip_delta,
                        'k': K
                    }
                    result.update(metrics)
                    pip_results.append(result)
                    
                    print(f"-> Recall@{K}={metrics[f'Recall@{K}']:.4f}, QPS={metrics['QPS']:.2f}")

pip_df = pd.DataFrame(pip_results)
print(f"\nPiP experiments completed: {len(pip_df)} configurations")

---

## 8. Ada-ef

In [ ]:
print("Running Ada-ef experiments...")

adaef_results = []

M_values = [16]
efC_values = [100, 200]
target_recall_values = [0.85, 0.90, 0.95]

for M in M_values:
    for efC in efC_values:
        for target_recall in target_recall_values:
            print(f"  Ada-ef: M={M}, efC={efC}, target_recall={target_recall}", end=' ')
            
            index = build_hnsw_adaef(
                xb, M=M, efC=efC,
                offline_k=K,
                offline_target_recall=target_recall
            )
            sf = hnsw_adaef_search_fn(index, target_recall=target_recall)
            metrics = measure_search(sf, xq, K, I_gt)
            
            result = {
                'Method': 'Ada-ef',
                'M': M,
                'efConstruction': efC,
                'target_recall': target_recall,
                'k': K
            }
            result.update(metrics)
            adaef_results.append(result)
            
            print(f"-> Recall@{K}={metrics[f'Recall@{K}']:.4f}, QPS={metrics['QPS']:.2f}")

adaef_df = pd.DataFrame(adaef_results)
print(f"\nAda-ef experiments completed: {len(adaef_df)} configurations")

---

## 9. Merge All Results

In [ ]:
all_results = pd.concat([hnsw_df, darth_df, pip_df, adaef_df], ignore_index=True)

print("=== All Results Merged ===")
print(f"Total configurations: {len(all_results)}")
print(f"\nMethods included: {all_results['Method'].unique().tolist()}")
print(f"\nColumns: {all_results.columns.tolist()}")

all_results.head(10)

In [ ]:
MASTER_CSV = RESULTS_CSV / '05_unified_comparison_master.csv'
all_results.to_csv(MASTER_CSV, index=False)
print(f"Master results saved to: {MASTER_CSV}")

---

## 10. Summary Tables

In [ ]:
def create_summary_table(df, metric_col, maximize=True):
    """Create summary table showing best configuration per method."""
    summary = []
    
    for method in df['Method'].unique():
        method_df = df[df['Method'] == method].copy()
        
        if maximize:
            best_idx = method_df[metric_col].idxmax()
        else:
            best_idx = method_df[metric_col].idxmin()
        
        best_row = method_df.loc[best_idx]
        summary.append(best_row)
    
    return pd.DataFrame(summary)


print("=== Best Configuration per Method (by Recall) ===")
best_by_recall = create_summary_table(all_results, f'Recall@{K}', maximize=True)
display_cols = ['Method', 'M', 'efConstruction', f'Recall@{K}', 'QPS', 'Avg Latency (ms)']
available_cols = [c for c in display_cols if c in best_by_recall.columns]
print(best_by_recall[available_cols].to_string(index=False))

print("\n\n=== Best Configuration per Method (by QPS) ===")
best_by_qps = create_summary_table(all_results, 'QPS', maximize=True)
print(best_by_qps[available_cols].to_string(index=False))

In [ ]:
RANKED_CSV = RESULTS_CSV / '05_unified_comparison_ranked.csv'
best_by_recall.to_csv(RANKED_CSV, index=False)
print(f"Ranked summary saved to: {RANKED_CSV}")

In [ ]:
print("=== Complete Results Table ===")
display_cols_all = ['Method', 'M', 'efConstruction', 'efSearch', f'Recall@{K}', 
                    'QPS', 'Avg Latency (ms)', 'Build Time (s)']
available_cols_all = [c for c in display_cols_all if c in all_results.columns]

pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)
print(all_results[available_cols_all].sort_values([f'Recall@{K}'], ascending=False).to_string(index=False))

---

## 11. Unified Comparison Plots

In [ ]:
METHOD_COLORS = {
    'HNSW': '#1f77b4',
    'DARTH': '#ff7f0e',
    'PiP': '#2ca02c',
    'Ada-ef': '#d62728'
}

METHOD_MARKERS = {
    'HNSW': 'o',
    'DARTH': 's',
    'PiP': '^',
    'Ada-ef': 'D'
}

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

for method in ['HNSW', 'DARTH', 'PiP', 'Ada-ef']:
    method_data = all_results[all_results['Method'] == method]
    if len(method_data) == 0:
        continue
    
    ax.scatter(
        method_data[f'Recall@{K}'],
        method_data['QPS'],
        c=METHOD_COLORS[method],
        marker=METHOD_MARKERS[method],
        s=100,
        label=method,
        alpha=0.7,
        edgecolors='black',
        linewidths=0.5
    )

ax.set_xlabel(f'Recall@{K}')
ax.set_ylabel('QPS (queries per second)')
ax.set_title(f'Recall@{K} vs QPS - All Methods Comparison')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_RESULTS / '05_recall_vs_qps_all_methods.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {PLOT_RESULTS / '05_recall_vs_qps_all_methods.png'}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

for method in ['HNSW', 'DARTH', 'PiP', 'Ada-ef']:
    method_data = all_results[all_results['Method'] == method]
    if len(method_data) == 0:
        continue
    
    ax.scatter(
        method_data[f'Recall@{K}'],
        method_data['Avg Latency (ms)'],
        c=METHOD_COLORS[method],
        marker=METHOD_MARKERS[method],
        s=100,
        label=method,
        alpha=0.7,
        edgecolors='black',
        linewidths=0.5
    )

ax.set_xlabel(f'Recall@{K}')
ax.set_ylabel('Average Latency (ms)')
ax.set_title(f'Recall@{K} vs Latency - All Methods Comparison')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_RESULTS / '05_recall_vs_latency_all_methods.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {PLOT_RESULTS / '05_recall_vs_latency_all_methods.png'}")

In [ ]:
best_per_method = best_by_recall.copy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

methods = best_per_method['Method'].unique()
colors = [METHOD_COLORS.get(m, 'gray') for m in methods]

axes[0].bar(methods, best_per_method[f'Recall@{K}'], color=colors, edgecolor='black')
axes[0].set_ylabel(f'Recall@{K}')
axes[0].set_title('Best Recall per Method')
axes[0].set_ylim([0, 1.05])
for i, v in enumerate(best_per_method[f'Recall@{K}']):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)

axes[1].bar(methods, best_per_method['QPS'], color=colors, edgecolor='black')
axes[1].set_ylabel('QPS')
axes[1].set_title('Best QPS per Method')
for i, v in enumerate(best_per_method['QPS']):
    axes[1].text(i, v + max(best_per_method['QPS'])*0.02, f'{v:.1f}', ha='center', fontsize=9)

axes[2].bar(methods, best_per_method['Avg Latency (ms)'], color=colors, edgecolor='black')
axes[2].set_ylabel('Avg Latency (ms)')
axes[2].set_title('Best Latency per Method')
for i, v in enumerate(best_per_method['Avg Latency (ms)']):
    axes[2].text(i, v + max(best_per_method['Avg Latency (ms)'])*0.02, f'{v:.2f}', ha='center', fontsize=9)

for ax in axes:
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(PLOT_RESULTS / '05_best_per_method_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {PLOT_RESULTS / '05_best_per_method_bar.png'}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, method in enumerate(['HNSW', 'DARTH', 'PiP', 'Ada-ef']):
    ax = axes[idx // 2, idx % 2]
    method_data = all_results[all_results['Method'] == method].sort_values(f'Recall@{K}')
    
    if len(method_data) == 0:
        continue
    
    ax.plot(method_data[f'Recall@{K}'], method_data['QPS'], 
            marker='o', color=METHOD_COLORS[method], linewidth=2, markersize=6)
    
    ax.set_xlabel(f'Recall@{K}')
    ax.set_ylabel('QPS')
    ax.set_title(f'{method}: Recall vs QPS')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_RESULTS / '05_recall_qps_per_method.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {PLOT_RESULTS / '05_recall_qps_per_method.png'}")

---

## 12. Final Analysis and Conclusions

In [ ]:
print("=" * 80)
print("THESIS-STYLE ANALYSIS: UNIFIED COMPARISON OF ANNS METHODS")
print("=" * 80)

high_recall_threshold = 0.95
high_recall_df = all_results[all_results[f'Recall@{K}'] >= high_recall_threshold]

print(f"\n1. BEST METHOD UNDER HIGH RECALL (>= {high_recall_threshold})\n")
if len(high_recall_df) > 0:
    best_high_recall = high_recall_df.loc[high_recall_df['QPS'].idxmax()]
    print(f"   Method: {best_high_recall['Method']}")
    print(f"   Recall@{K}: {best_high_recall[f'Recall@{K}']:.4f}")
    print(f"   QPS: {best_high_recall['QPS']:.2f}")
    print(f"   Configuration: {dict(best_high_recall[['M', 'efConstruction', 'efSearch']])}")
else:
    print("   No method achieved the target recall threshold.")
    print("   Showing top performers by recall:")
    top_recall = all_results.nlargest(3, f'Recall@{K}')
    for _, row in top_recall.iterrows():
        print(f"   - {row['Method']}: Recall={row[f'Recall@{K}']:.4f}, QPS={row['QPS']:.2f}")

print(f"\n2. BEST METHOD UNDER SPEED CONSTRAINTS (highest QPS)\n")
best_qps_row = all_results.loc[all_results['QPS'].idxmax()]
print(f"   Method: {best_qps_row['Method']}")
print(f"   QPS: {best_qps_row['QPS']:.2f}")
print(f"   Recall@{K}: {best_qps_row[f'Recall@{K}']:.4f}")
print(f"   Configuration: {dict(best_qps_row[['M', 'efConstruction', 'efSearch']])}")

print(f"\n3. TRADE-OFF ANALYSIS (Recall-Speed Pareto Frontier)\n")
def find_pareto_frontier(df, recall_col, qps_col):
    pareto = []
    sorted_df = df.sort_values(recall_col, ascending=False)
    max_qps = 0
    for _, row in sorted_df.iterrows():
        if row[qps_col] > max_qps:
            pareto.append(row)
            max_qps = row[qps_col]
    return pd.DataFrame(pareto)

pareto_df = find_pareto_frontier(all_results, f'Recall@{K}', 'QPS')
print("   Pareto-optimal configurations (by Recall@{K}):")
for _, row in pareto_df.iterrows():
    print(f"   - {row['Method']}: Recall={row[f'Recall@{K}']:.4f}, QPS={row['QPS']:.2f}")

print(f"\n4. STABILITY ANALYSIS (variance in recall across configurations)\n")
for method in all_results['Method'].unique():
    method_data = all_results[all_results['Method'] == method]
    recall_std = method_data[f'Recall@{K}'].std()
    recall_mean = method_data[f'Recall@{K}'].mean()
    qps_std = method_data['QPS'].std()
    qps_mean = method_data['QPS'].mean()
    print(f"   {method}:")
    print(f"     Recall: mean={recall_mean:.4f}, std={recall_std:.4f}")
    print(f"     QPS: mean={qps_mean:.2f}, std={qps_std:.2f}")

In [ ]:
print("\n" + "=" * 80)
print("SUMMARY COMMENTS")
print("=" * 80)

print("Q1: Which method gives the best recall-speed tradeoff?")

if len(pareto_df) > 0:
    pareto_methods = pareto_df['Method'].unique()
    print(f"    Methods on Pareto frontier: {', '.join(pareto_methods)}")
else:
    print("    Check individual method results for trade-off characteristics.")

print("Q2: Which method reduces unnecessary search effort?")

min_latency = all_results['Avg Latency (ms)'].min()
best_efficient = all_results[all_results['Avg Latency (ms)'] == min_latency].iloc[0]
print(f"    Lowest average latency: {best_efficient['Method']} ({min_latency:.3f} ms)")

print("Q3: Which method is most stable across the workload?")

stability = all_results.groupby('Method')[f'Recall@{K}'].std()
most_stable = stability.idxmin()
print(f"    Most stable (lowest recall variance): {most_stable} (std={stability[most_stable]:.4f})")

print("""
NOTES:
- Results depend on the specific dataset, parameters, and hardware.
- DARTH and PiP aim to reduce search effort via early termination.
- Ada-ef adapts exploration factor based on estimated query difficulty.
- Consider the specific application requirements when choosing a method.
""")

In [ ]:
print("\n" + "=" * 80)
print("OUTPUT FILES GENERATED")
print("=" * 80)
print(f"\n1. Master results CSV: {MASTER_CSV}")
print(f"2. Ranked summary CSV: {RANKED_CSV}")
print(f"\n3. Plots:")
print(f"   - {PLOT_RESULTS / '05_recall_vs_qps_all_methods.png'}")
print(f"   - {PLOT_RESULTS / '05_recall_vs_latency_all_methods.png'}")
print(f"   - {PLOT_RESULTS / '05_best_per_method_bar.png'}")
print(f"   - {PLOT_RESULTS / '05_recall_qps_per_method.png'}")
print("\n" + "=" * 80)
print("NOTEBOOK COMPLETED SUCCESSFULLY")
print("=" * 80)